Phase 4

In [2]:
# 1. Environment, CONFIG, hardware
import os, gc, sys, json, math, time, random, glob, zipfile, pickle, warnings, re, itertools, hashlib
import types as _types
from functools import partial
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.checkpoint import checkpoint
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

DRY_RUN = False

# ------------------------------- CONFIG ---------------------------------------
RUNS_TO_EXECUTE = [1, 2, 3, 4, 5]
PREV_OUT_DIRS = []          # e.g. ['/kaggle/input/cjpe-v4-part1/preds'] to reuse earlier sessions' results

# Backbone geometry (must match the Phase-1 checkpoints)
D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS = 768, 12, 12, 4
FF_SWIGLU, FF_GELU, MAX_POS = 2560, 3072, 2048

# Text pre-tokenisation before BPE. 'auto' = decided by the vocabulary check + perplexity grid in cell 7.
# Manual options: 'split' (v3 behaviour), 'lower', 'punct' (punctuation split off words), 'punct_lower'.
PREPROC = 'auto'

# Document view (identical at train and eval time)
CHUNK_LEN   = 512
HEAD_CHUNKS = 1             # opening of the judgment: facts, issues
TAIL_CHUNKS = 3             # last 1,536 tokens: the court's reasoning
ENC_BS      = 16            # chunks per backbone forward pass while fine-tuning (= 4 docs)
EXTRACT_BS  = 32            # chunks per forward pass during no-grad feature extraction
POOL_LAYERS = (3, 6, 9)     # outputs of blocks 4, 7, 10 are mean-pooled alongside the final layer
GRAD_CKPT   = True

# Stage A (head on cached frozen features)
LP_EPOCHS, LP_BS, LP_LR, LP_WD, LP_PATIENCE = 30, 64, 5e-4, 0.05, 5
HEAD_DROP = 0.3

# Stage B (PEFT fine-tuning; starts from the Stage-A head, adapters start as the identity)
FT_MINUTES    = {1: 75, 2: 45, 3: 45, 4: 40}   # wall-clock budget per run (0 = skip Stage B)
FT_DOCS_PER_FWD = 4         # documents per forward pass
FT_ACCUM      = 16          # documents per optimiser step
FT_LR_PEFT    = 2e-4
FT_LR_HEAD    = 1e-4
FT_EVAL_EVERY = 3000        # documents between full-Dev evaluations
FT_PATIENCE   = 3
LR_LAYER_TOP, XI = 5e-5, 0.85   # Run 3: LR of the top block, geometric decay per block below it
KD_DOCS       = 12000       # training docs whose teacher outputs are cached for Run 4
KD_ALPHA, T_KD = 0.5, 2.0   # Run 4: loss = KD_ALPHA * KL(T) + (1 - KD_ALPHA) * CE

LABEL_SMOOTH  = 0.05
CACHE_FEATS_TO_DISK = True
USE_FUSION_MODEL = True     # Run 5: linear model on concatenated frozen features of all four LMs
USE_TFIDF_MODEL = False     # classical TF-IDF model as an extra member (only if the rules allow it)
SEED = 42
DRY_RUN_TRAIN_DOCS = 2000
if DRY_RUN:
    LP_EPOCHS, FT_EVAL_EVERY, KD_DOCS = 8, 400, 500
    FT_MINUTES = {1: 4, 2: 3, 3: 3, 4: 3}
    CACHE_FEATS_TO_DISK = False

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16
AMP_ON = True               # switched off per run if the backbone overflows in fp16 (checked automatically)
print(f"Device: {device} | AMP: {USE_AMP} ({AMP_DTYPE})")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

def amp_ctx():
    on = USE_AMP and AMP_ON
    return torch.autocast(device_type=device.type, dtype=AMP_DTYPE if on else torch.bfloat16, enabled=on)

def make_scaler():
    on = USE_AMP and AMP_ON and AMP_DTYPE == torch.float16
    try:
        return torch.amp.GradScaler('cuda', enabled=on)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=on)

def free_mem():
    for name in ['_', '__', '___']:
        if name in globals():
            globals()[name] = None
    for a in ('last_traceback', 'last_value', 'last_type'):
        if hasattr(sys, a): setattr(sys, a, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./working"
OUT_DIR = os.path.join(WORK_DIR, "preds")
FEAT_DIR = os.path.join(WORK_DIR, "feats")
for d in (WORK_DIR, OUT_DIR, FEAT_DIR): os.makedirs(d, exist_ok=True)

# Hugging Face authentication: Kaggle secret 'HF_TOKEN' (Add-ons -> Secrets) or the HF_TOKEN env var.
# (The token that was hard-coded in v3 should be revoked at huggingface.co/settings/tokens.)
HF_TOKEN = None
for _name in ("HF_TOKEN", "HF"):
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret(_name)
        if HF_TOKEN: break
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN","")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN); print("✓ Hugging Face authentication successful.")
    except Exception as e:
        print(f"HF login note: {e}")
else:
    print("No HF token found (fine if the dataset is public or attached as a Kaggle input).")

Device: cuda | AMP: True (torch.float16)
GPU: Tesla T4 (15.6 GB)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Hugging Face authentication successful.


In [3]:
# 2. Universal Checkpoint Resolver & Directory-to-.PT Converter

def find_checkpoint(candidates: List[str], glob_pattern: str = "") -> Optional[str]:
    for c in candidates:
        if os.path.exists(c):
            return c
    if glob_pattern:
        matches = sorted(glob.glob(glob_pattern, recursive=True))
        if matches:
            return matches[0]
    return None

def pack_directory_to_pt(source_dir: str, target_pt: str) -> str:
    print(f"-> Directory detected: '{source_dir}'. Repacking to valid .pt archive: '{target_pt}'...")
    files = [(os.path.join(r, f), os.path.relpath(os.path.join(r, f), source_dir))
             for r, _, fs in os.walk(source_dir) for f in fs]
    has_top_dir = any(rel.endswith('data.pkl') and os.sep in rel for _, rel in files)
    prefix = '' if has_top_dir else 'archive/'
    with zipfile.ZipFile(target_pt, 'w', compression=zipfile.ZIP_STORED) as zf:
        for full, rel in files:
            zf.write(full, arcname=prefix + rel)
    print(f"✓ Created checkpoint archive: {target_pt} ({os.path.getsize(target_pt)/1e6:.1f} MB)")
    return target_pt

ARCH4_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt',
    '/kaggle/input/cs781-phase1-arch4/decoder_lm.pt',
    '/kaggle/input/phase1-arch4/decoder_lm.pt',
    './decoder_lm.pt',
    'd:/D_Downloads/Phase-1/decoder_lm.pt',
    'd:/D_Downloads/decoder_lm.pt'
]
ARCH3_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch3/decoder_lm',
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch3/decoder_lm.pt',
    '/kaggle/input/cs781-phase1-arch3/decoder_lm.pt',
    '/kaggle/input/phase1-arch3/decoder_lm.pt',
    './decoder_lm_arch3.pt'
]
ARCH2_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-arch2-phase1/decoder_lm.pt',
    '/kaggle/input/cs781-phase1-arch2/decoder_lm.pt',
    '/kaggle/input/phase1-arch2/decoder_lm.pt',
    './decoder_lm_arch2.pt',
    'd:/D_Downloads/Phase-1/decoder_lm.pt'
]
ARCH1_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-legal-model-checkpoints/decoder_lm.pt',
    '/kaggle/input/cs781-legal-model-checkpoints/decoder_lm.pt',
    '/kaggle/input/phase1-arch1/decoder_lm.pt',
    './decoder_lm_arch1.pt'
]
TOKENIZER_CANDIDATES = [
    '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json',
    '/kaggle/input/custom_bpe.json',
    './custom_bpe.json',
    'd:/D_Downloads/Phase-1/custom_bpe.json'
]

ARCH4_RAW = find_checkpoint(ARCH4_CANDIDATES, "/kaggle/input/**/cs781-phase1-arch4/**/decoder_lm*") or find_checkpoint([], "/kaggle/input/**/decoder_lm.pt")
ARCH3_RAW = find_checkpoint(ARCH3_CANDIDATES, "/kaggle/input/**/cs781-phase1-arch3/**")
ARCH2_RAW = find_checkpoint(ARCH2_CANDIDATES, "/kaggle/input/**/cs781-phase1-arch2/**")
ARCH1_RAW = find_checkpoint(ARCH1_CANDIDATES, "/kaggle/input/**/phase1-arch1/**")
TOKENIZER_PATH = find_checkpoint(TOKENIZER_CANDIDATES, "/kaggle/input/**/custom_bpe.json")

print("Discovered Raw Checkpoint Paths:")
print(f"  Arch 4 Raw : {ARCH4_RAW}")
print(f"  Arch 3 Raw : {ARCH3_RAW}")
print(f"  Arch 2 Raw : {ARCH2_RAW}")
print(f"  Arch 1 Raw : {ARCH1_RAW}")
print(f"  Tokenizer  : {TOKENIZER_PATH}")

ARCH4_CHECKPOINT = ARCH4_RAW
if ARCH4_RAW and os.path.isdir(ARCH4_RAW):
    ARCH4_CHECKPOINT = pack_directory_to_pt(ARCH4_RAW, os.path.join(WORK_DIR, 'decoder_lm_arch4.pt'))

ARCH3_CHECKPOINT = ARCH3_RAW
if ARCH3_RAW and os.path.isdir(ARCH3_RAW):
    ARCH3_CHECKPOINT = pack_directory_to_pt(ARCH3_RAW, os.path.join(WORK_DIR, 'decoder_lm_arch3.pt'))

ARCH2_CHECKPOINT = ARCH2_RAW
if ARCH2_RAW and os.path.isdir(ARCH2_RAW):
    ARCH2_CHECKPOINT = pack_directory_to_pt(ARCH2_RAW, os.path.join(WORK_DIR, 'decoder_lm_arch2.pt'))

ARCH1_CHECKPOINT = ARCH1_RAW
if ARCH1_RAW and os.path.isdir(ARCH1_RAW):
    ARCH1_CHECKPOINT = pack_directory_to_pt(ARCH1_RAW, os.path.join(WORK_DIR, 'decoder_lm_arch1.pt'))

Discovered Raw Checkpoint Paths:
  Arch 4 Raw : /kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt
  Arch 3 Raw : /kaggle/input/datasets/brindafadadu/cs781-phase1-arch3/decoder_lm
  Arch 2 Raw : /kaggle/input/datasets/brindafadadu/cs781-arch2-phase1/decoder_lm.pt
  Arch 1 Raw : /kaggle/input/datasets/brindafadadu/cs781-legal-model-checkpoints/decoder_lm.pt
  Tokenizer  : /kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json
-> Directory detected: '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt'. Repacking to valid .pt archive: '/kaggle/working/decoder_lm_arch4.pt'...
✓ Created checkpoint archive: /kaggle/working/decoder_lm_arch4.pt (421.1 MB)
-> Directory detected: '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch3/decoder_lm'. Repacking to valid .pt archive: '/kaggle/working/decoder_lm_arch3.pt'...
✓ Created checkpoint archive: /kaggle/working/decoder_lm_arch3.pt (330.1 MB)


In [4]:
# 3. Byte-level BPE tokenizer + selectable pre-tokenisation
#    The BPE merges only reproduce Phase 1 if the text is split into "words" the same way Phase 1 did.
#    v3 used text.split(); if Phase 1 lower-cased the text or split punctuation off words, every
#    word like "Court," or "Appellant" becomes an unusual token sequence and perplexity explodes.
SPECIALS = ['<pad>', '<unk>', '<s>', '</s>']
END_OF_WORD = '</w>'
_PUNCT_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
PREPROC_MODES = ('split', 'lower', 'punct', 'punct_lower')

def pretokenize(text: str, mode: str) -> List[str]:
    if mode in ('lower', 'punct_lower'):
        text = text.lower()
    if mode in ('punct', 'punct_lower'):
        return _PUNCT_RE.findall(text)
    return text.split()

class FastByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges, self.merge_rank, self.itos, self.stoi = [], {}, [], {}
        self._word_cache = {}

    @staticmethod
    def _word_to_symbols(word):
        return list(word.encode('utf-8').decode('latin-1')) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        s = self._word_to_symbols(word)
        while len(s) > 1:
            p = [(s[i], s[i+1]) for i in range(len(s)-1)]
            c = [(self.merge_rank[x], i) for i, x in enumerate(p) if x in self.merge_rank]
            if not c: break
            _, i = min(c)
            s = s[:i] + [s[i]+s[i+1]] + s[i+2:]
        return s

    def _word_ids(self, w):
        r = self._word_cache.get(w)
        if r is None:
            unk = self.stoi.get('<unk>', 1)
            r = [self.stoi.get(s, unk) for s in self._apply_merges_to_word(w)]
            self._word_cache[w] = r
        return r

    def encode_words(self, words):
        return [i for w in words for i in self._word_ids(w)]

    def encode(self, text, max_len=None, mode='split'):
        ids = self.encode_words(pretokenize(text, mode))
        return ids[:max_len] if max_len else ids

    @classmethod
    def load(cls, path):
        with open(path, 'r', encoding='utf-8') as f:
            d = json.load(f)
        t = cls(vocab_size=d['vocab_size'])
        t.itos = d['itos']
        t.stoi = {x: i for i, x in enumerate(t.itos)}
        t.merges = [tuple(p) for p in d['merges']]
        t.merge_rank = {p: i for i, p in enumerate(t.merges)}
        return t

if not TOKENIZER_PATH or not os.path.exists(TOKENIZER_PATH):
    try:
        from huggingface_hub import hf_hub_download
        TOKENIZER_PATH = hf_hub_download(repo_id="Exploration-Lab/CS-781-Capstone", filename="Phase_1_LM/custom_bpe.json", repo_type="dataset", token=HF_TOKEN)
    except Exception as e:
        print(f"Tokenizer download note: {e}")

tokenizer = FastByteBPETokenizer.load(TOKENIZER_PATH)
PAD_ID = tokenizer.stoi.get('<pad>', 0)
SEP_ID = tokenizer.stoi.get('</s>', 3)
UNK_ID = tokenizer.stoi.get('<unk>', 1)
print(f"✓ Tokenizer ready: {tokenizer.vocab_size} vocab tokens | PAD_ID={PAD_ID}, SEP_ID={SEP_ID}")

def inspect_vocab(tok) -> Dict[str, float]:
    """What does the vocabulary say about how Phase 1 pre-tokenised its text?"""
    body = [t.replace(END_OF_WORD, '') for t in tok.itos if t not in SPECIALS]
    body = [b for b in body if len(b) > 1]                          # skip single bytes
    n = max(1, len(body))
    upper = sum(any('A' <= c <= 'Z' for c in b) for b in body) / n
    glued = sum(bool(re.search(r"[A-Za-z0-9][.,;:()\"'\]\[]|[.,;:()\"'\]\[][A-Za-z0-9]", b)) for b in body) / n
    ex_up = [b for b in body if any('A' <= c <= 'Z' for c in b)][:8]
    ex_gl = [b for b in body if re.search(r"[A-Za-z0-9][.,;:()]|[.,;:()][A-Za-z0-9]", b)][:8]
    print(f"Vocabulary check: {upper*100:.2f}% of multi-byte tokens contain upper-case letters {ex_up} | "
          f"{glued*100:.2f}% glue punctuation to letters/digits {ex_gl}")
    hint = ('punct' if glued < 0.002 else '') + ('_lower' if upper < 0.005 else '')
    hint = {'': 'split', '_lower': 'lower', 'punct': 'punct', 'punct_lower': 'punct_lower'}[hint]
    print(f"  -> the vocabulary suggests pre-tokeniser '{hint}' (confirmed by perplexity in cell 7)")
    return {'upper_frac': upper, 'glued_frac': glued, 'hint': hint}

VOCAB_INFO = inspect_vocab(tokenizer)

✓ Tokenizer ready: 8000 vocab tokens | PAD_ID=0, SEP_ID=3
Vocabulary check: 19.12% of multi-byte tokens contain upper-case letters ['Th', 'The', 'Ac', 'Cour', 'Court', 'In', 'Act', 'It'] | 15.10% glue punctuation to letters/digits ['s.', 'e,', 't.', 't,', '(1', 's,', 'y,', 'e.']
  -> the vocabulary suggests pre-tokeniser 'split' (confirmed by perplexity in cell 7)


In [5]:
# 4. Phase 4 CJPE Dataset Loading & Class Balancing (unchanged except the DRY_RUN subset)
def get_dataset_file(filename: str) -> str:
    local_hits = glob.glob(f"/kaggle/input/**/{Path(filename).name}", recursive=True)
    if local_hits:
        return local_hits[0]
    local_hits2 = glob.glob(f"**/{Path(filename).name}", recursive=True)
    if local_hits2:
        return local_hits2[0]
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id="Exploration-Lab/CS-781-Capstone", filename=filename, repo_type="dataset", token=HF_TOKEN)

print("Resolving Phase 4 CJPE dataset files...")
TRAIN_FILE = get_dataset_file("Phase_4_CJPE/Train/train.json")
DEV_FILE   = get_dataset_file("Phase_4_CJPE/Train/dev.json")
TEST_FILE  = get_dataset_file("Phase_4_CJPE/Test/test.json")
print(f"  Train : {TRAIN_FILE}")
print(f"  Dev   : {DEV_FILE}")
print(f"  Test  : {TEST_FILE}")

def load_cjpe_data(file_path: str, is_test: bool = False) -> List[Dict[str, Any]]:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    if isinstance(data, dict):
        if 'data' in data and isinstance(data['data'], list):
            raw_list = data['data']
        else:
            raw_list = [{'id': k, **v} if isinstance(v, dict) else {'id': k, 'text': v} for k, v in data.items()]
    elif isinstance(data, list):
        raw_list = data
    else:
        raise ValueError(f"Unsupported format in {file_path}")

    cleaned = []
    for idx, item in enumerate(raw_list):
        cid = str(item.get('id', item.get('case_id', f'case_{idx:04d}')))
        text = str(item.get('text', item.get('case_text', ''))).strip()
        label = item.get('label', item.get('judgment', item.get('outcome', None)))
        if not is_test and label is not None:
            label = int(label)
        cleaned.append({'id': cid, 'text': text, 'label': label})
    return cleaned

train_records = load_cjpe_data(TRAIN_FILE, is_test=False)
dev_records   = load_cjpe_data(DEV_FILE, is_test=False)
test_records  = load_cjpe_data(TEST_FILE, is_test=True)

if DRY_RUN:
    train_records = random.Random(SEED).sample(train_records, min(DRY_RUN_TRAIN_DOCS, len(train_records)))
    print(f"DRY_RUN: training on a random subset of {len(train_records)} documents.")

print(f"Loaded records -> Train: {len(train_records)} | Dev: {len(dev_records)} | Test: {len(test_records)}")

train_labels = [r['label'] for r in train_records if r['label'] is not None]
counts = Counter(train_labels)
print(f"Train Class Distribution: Class 0 = {counts.get(0, 0)} ({counts.get(0, 0)/len(train_labels)*100:.1f}%), Class 1 = {counts.get(1, 0)} ({counts.get(1, 0)/len(train_labels)*100:.1f}%)")

total_samples = len(train_labels)
w0 = total_samples / (2.0 * max(1, counts.get(0, 1)))
w1 = total_samples / (2.0 * max(1, counts.get(1, 1)))
class_weights = torch.tensor([w0, w1], dtype=torch.float32).to(device)
print(f"Balanced Class Weights: Class 0 = {w0:.3f}, Class 1 = {w1:.3f}")

Resolving Phase 4 CJPE dataset files...


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

  Train : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/train.json
  Dev   : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/dev.json
  Test  : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Test/test.json
Loaded records -> Train: 37387 | Dev: 1988 | Test: 1517
Train Class Distribution: Class 0 = 22067 (59.0%), Class 1 = 15320 (41.0%)
Balanced Class Weights: Class 0 = 0.847, Class 1 = 1.220


In [6]:
# 5. All four Phase-1 architectures (defined up front so every later cell can use them)
#    Forward-pass details that the checkpoints do not pin down are switchable via ARCH_OPTS;
#    cell 7 picks the variant with the lowest perplexity.
ARCH_OPTS = {
    1: {'act': 'gelu'},                                          # Arch 1 FFN activation
    2: {'rope': 'half', 'qkn': 'after', 'swap': False},          # Arch 2 RoPE pairing / qk-norm order / SwiGLU branch
    3: {'kv': 'interleave', 'act': 'relu'},                      # Arch 3 kv-head sharing / FFN activation
}

def _act(h, name):
    if name == 'gelu': return F.gelu(h)
    if name == 'gelu_tanh': return F.gelu(h, approximate='tanh')
    if name == 'silu': return F.silu(h)
    return F.relu(h)

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)

# ---------------------------- Arch 4: ALiBi + SwiGLU + RMSNorm ----------------
def get_alibi_slopes(n_heads: int) -> torch.Tensor:
    def p2(n):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        return [start * (start ** i) for i in range(n)]
    if math.log2(n_heads).is_integer():
        return torch.tensor(p2(n_heads))
    c = 2 ** math.floor(math.log2(n_heads))
    return torch.tensor(p2(c) + p2(2 * c)[0::2][:n_heads - c])

class ALiBiCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.d_model, self.n_heads, self.head_dim, self.dropout = d_model, n_heads, d_model // n_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)
        self.register_buffer("slopes", get_alibi_slopes(n_heads).unsqueeze(1).unsqueeze(2))
        self._bias_cache = {}
    def _bias(self, S, dtype, dev):
        key = (S, dtype, str(dev))
        b = self._bias_cache.get(key)
        if b is None:
            pos = torch.arange(S, device=dev)
            alibi = self.slopes.to(dev).float() * (pos[None, :] - pos[:, None]).float().unsqueeze(0)
            causal = torch.triu(torch.full((S, S), float('-inf'), device=dev), diagonal=1)
            b = (alibi + causal).unsqueeze(0).to(dtype)
            self._bias_cache = {key: b}
        return b
    def forward(self, x):
        B, S, _ = x.shape
        q = self.q_norm(self.q_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2))
        k = self.k_norm(self.k_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2))
        v = self.v_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=self._bias(S, q.dtype, x.device),
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, S, self.d_model))

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, arch=None):
        super().__init__()
        self.arch = arch
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout   = nn.Dropout(dropout)
    def forward(self, x):
        g, u = self.gate_proj(x), self.up_proj(x)
        swap = self.arch is not None and ARCH_OPTS[self.arch].get('swap', False)
        return self.dropout(self.down_proj(F.silu(u) * g if swap else F.silu(g) * u))

class TransformerBlockArch4(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model); self.attn = ALiBiCausalAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model); self.ffn = SwiGLU(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

class ALiBiSwiGLULM(nn.Module):
    def __init__(self, vocab_size=8000, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS, d_ff=FF_SWIGLU, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlockArch4(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
    def forward(self, idx):
        x = self.drop(self.tok_emb(idx))
        for block in self.blocks: x = block(x)
        return self.norm_f(x)

# ---------------------------- Arch 3: GQA + learned positions + LayerNorm -----
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, n_kv_heads=N_KV_HEADS, dropout=0.1):
        super().__init__()
        self.d_model, self.n_heads, self.n_kv_heads = d_model, n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.num_queries_per_kv = n_heads // n_kv_heads
        self.dropout = dropout
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model, bias=False)
    def forward(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        g = self.num_queries_per_kv
        if ARCH_OPTS[3]['kv'] == 'interleave':
            k, v = k.repeat_interleave(g, dim=1), v.repeat_interleave(g, dim=1)
        else:
            k, v = k.repeat(1, g, 1, 1), v.repeat(1, g, 1, 1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, self.d_model))

class FeedForwardArch3(nn.Module):
    def __init__(self, d_model=D_MODEL, d_ff=FF_GELU, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff); self.fc2 = nn.Linear(d_ff, d_model); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.fc2(_act(self.fc1(x), ARCH_OPTS[3]['act'])))

class GQABlock(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, n_kv_heads=N_KV_HEADS, d_ff=FF_GELU, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model); self.attn = GroupedQueryAttention(d_model, n_heads, n_kv_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model); self.ffn = FeedForwardArch3(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))

class GQADecoderLM(nn.Module):
    def __init__(self, vocab_size=8000, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
                 n_kv_heads=N_KV_HEADS, d_ff=FF_GELU, max_seq_len=MAX_POS, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([GQABlock(d_model, n_heads, n_kv_heads, d_ff, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks: x = block(x)
        return self.ln_f(x)

# ---------------------------- Arch 2: RoPE + SwiGLU + RMSNorm -----------------
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=MAX_POS, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)
    def forward(self, x, seq_len):
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]

def rotate_half(x):
    h = x.shape[-1] // 2
    return torch.cat((-x[..., h:], x[..., :h]), dim=-1)

def _rot_pairs(x):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)

def apply_rope(q, k, cos, sin, mode):
    if mode == 'half':
        c, s = cos[None, None].to(q.dtype), sin[None, None].to(q.dtype)
        return q * c + rotate_half(q) * s, k * c + rotate_half(k) * s
    h = cos.shape[-1] // 2
    c = cos[..., :h].repeat_interleave(2, dim=-1)[None, None].to(q.dtype)
    s = sin[..., :h].repeat_interleave(2, dim=-1)[None, None].to(q.dtype)
    return q * c + _rot_pairs(q) * s, k * c + _rot_pairs(k) * s

class RoPECausalAttention(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, dropout=0.1):
        super().__init__()
        self.d_model, self.n_heads, self.head_dim, self.dropout = d_model, n_heads, d_model // n_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)
    def forward(self, x, cos, sin):
        B, S, _ = x.shape
        o = ARCH_OPTS[2]
        q = self.q_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        if o['qkn'] == 'before': q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, k, cos, sin, o['rope'])
        if o['qkn'] == 'after': q, k = self.q_norm(q), self.k_norm(k)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, S, self.d_model))

class RoPEBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=FF_SWIGLU, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model); self.attn = RoPECausalAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model); self.ffn = SwiGLU(d_model, d_ff, dropout, arch=2)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        return x + self.ffn(self.norm2(x))

class RoPESwiGLUDecoderLM(nn.Module):
    def __init__(self, vocab_size=8000, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
                 d_ff=FF_SWIGLU, max_seq_len=MAX_POS, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.rope = RotaryEmbedding(d_model // n_heads, max_seq_len=max_seq_len)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([RoPEBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
    def forward(self, idx):
        cos, sin = self.rope(idx, idx.size(1))
        x = self.drop(self.tok_emb(idx))
        for block in self.blocks: x = block(x, cos, sin)
        return self.norm_f(x)

# ---------------------------- Arch 1: sinusoidal + LayerNorm + GELU -----------
def sinusoidal_pos_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term); pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class CausalSelfAttentionArch1(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, dropout=0.1):
        super().__init__()
        self.d_model, self.n_heads, self.head_dim, self.dropout = d_model, n_heads, d_model // n_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, self.d_model))

class FeedForwardArch1(nn.Module):
    """net = [Linear, act, Dropout, Linear]: the Phase-1 checkpoint stores the two linears as net.0 / net.3."""
    def __init__(self, d_model=D_MODEL, d_ff=FF_GELU, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.Identity(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net[3](self.net[2](_act(self.net[0](x), ARCH_OPTS[1]['act'])))

class BlockArch1(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=FF_GELU, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model); self.attn = CausalSelfAttentionArch1(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model); self.ffn = FeedForwardArch1(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))

class StudentArch1LM(nn.Module):
    def __init__(self, vocab_size=8000, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
                 d_ff=FF_GELU, max_seq_len=MAX_POS, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pe", sinusoidal_pos_encoding(max_seq_len, d_model).unsqueeze(0), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([BlockArch1(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
    def forward(self, idx):
        x = self.drop(self.tok_emb(idx) + self.pe[:, :idx.size(1)].to(self.tok_emb.weight.dtype))
        for block in self.blocks: x = block(x)
        return self.ln_f(x)

print("✓ Architectures 1-4 defined.")

✓ Architectures 1-4 defined.


In [ ]:
# 6. Checkpoint loading (lenient: pickled config stand-ins, name aliases, block/shape matching) + PEFT adapters
# ---------------------------- checkpoint loading ------------------------------
_ALIASES = [('.n1.', '.norm1.'), ('.n2.', '.norm2.'), ('ln1', 'norm1'), ('ln2', 'norm2'),
            ('w_gate', 'gate_proj'), ('w_up', 'up_proj'), ('w_down', 'down_proj'),
            ('wq.', 'q_proj.'), ('wk.', 'k_proj.'), ('wv.', 'v_proj.'), ('wo.', 'out_proj.'), ('o_proj', 'out_proj'),
            ('final_norm', 'norm_f'), ('ln_f', 'norm_f'), ('embed_tokens', 'tok_emb'), ('wte', 'tok_emb'),
            ('layers.', 'blocks.'), ('self_attn', 'attn'), ('mlp.', 'ffn.')]
_PREFIXES = ('module.', '_orig_mod.', 'model.', 'backbone.', 'transformer.')

def _strip(k: str) -> str:
    changed = True
    while changed:
        changed = False
        for pre in _PREFIXES:
            if k.startswith(pre):
                k = k[len(pre):]; changed = True
    return k

def _key_variants(k: str):
    base = _strip(k); fwd, rev = base, base
    for a, b in _ALIASES:
        fwd = fwd.replace(a, b); rev = rev.replace(b, a)
    return [k, base, fwd, rev]

def disable_dropout(module: nn.Module):
    # Backbone dropout off: features are deterministic, and the frozen-feature head (Stage A)
    # sees exactly what the fine-tuned model produces at the start of Stage B.
    for m in module.modules():
        if isinstance(m, nn.Dropout): m.p = 0.0
        if hasattr(m, 'dropout') and isinstance(getattr(m, 'dropout'), float): m.dropout = 0.0

import types as _types, re as _re

class _PickleStub:
    def __init__(self, *a, **k):
        if k: self.__dict__.update(k)
    def __setstate__(self, state):
        if isinstance(state, dict): self.__dict__.update(state)
        elif isinstance(state, tuple) and len(state) == 2 and isinstance(state[1], dict): self.__dict__.update(state[1])
    def __repr__(self):
        return f"{type(self).__name__}({', '.join(f'{k}={v}' for k, v in self.__dict__.items() if not k.startswith('_'))})"

class _LenientUnpickler(pickle.Unpickler):
    def find_class(self, mod_name, name):
        try:
            return super().find_class(mod_name, name)
        except (AttributeError, ModuleNotFoundError, ImportError):
            print(f"   (checkpoint references {mod_name}.{name}, not defined here -> loaded as a stand-in)")
            return type(name, (_PickleStub,), {})

lenient_pickle = _types.ModuleType('lenient_pickle')
lenient_pickle.Unpickler = _LenientUnpickler
lenient_pickle.load = lambda f, **kw: _LenientUnpickler(f, **kw).load()
lenient_pickle.loads = pickle.loads

def torch_load_any(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except (AttributeError, ModuleNotFoundError, ImportError):
        return torch.load(path, map_location='cpu', weights_only=False, pickle_module=lenient_pickle)

def _block_id(k):
    m = _re.search(r'(?:^|\.)(?:blocks|layers|h|layer|decoder_layers)\.(\d+)\.', k)
    return int(m.group(1)) if m else None

def load_pretrained(backbone: nn.Module, path: Optional[str], name: str) -> float:
    if not path or not os.path.exists(path):
        print(f"⚠⚠ [{name}] checkpoint NOT FOUND -> backbone is RANDOMLY initialised (scores will be poor).")
        return 0.0
    raw = torch_load_any(path)
    if isinstance(raw, dict):
        for ck in ('config', 'cfg', 'model_config', 'args', 'hparams'):
            if ck in raw and not torch.is_tensor(raw[ck]):
                print(f"   [{name}] Phase-1 config: {raw[ck]}")
    sd = raw
    if isinstance(raw, dict):
        for key in ('model_state_dict', 'model', 'state_dict'):
            if key in raw and isinstance(raw[key], dict):
                sd = raw[key]; break
    sd = {k: v for k, v in sd.items() if torch.is_tensor(v)}
    model_sd = backbone.state_dict()
    param_names = [n for n, _ in backbone.named_parameters()]

    # (1) match by name (with prefix stripping / aliases)
    mapped, used = {}, set()
    for k, v in sd.items():
        for cand in _key_variants(k):
            if cand in model_sd and cand not in mapped and tuple(model_sd[cand].shape) == tuple(v.shape):
                mapped[cand] = v; used.add(k); break

    # (2) position tables of a different length: copy the overlapping rows
    for n in param_names:
        if n in mapped or 'pos' not in n: continue
        for k, v in sd.items():
            if k in used or 'pos' not in k or v.dim() != 2 or v.shape[1] != model_sd[n].shape[1]: continue
            t = model_sd[n].clone(); r = min(t.shape[0], v.shape[0]); t[:r] = v[:r]
            mapped[n] = t; used.add(k)
            print(f"   [{name}] {k} {tuple(v.shape)} -> {n} {tuple(t.shape)} (first {r} rows copied)")
            break

    # (3) remaining params: match by (block index, weight/bias, shape), in order of appearance
    groups_m, groups_c = {}, {}
    for n in param_names:
        if n not in mapped:
            groups_m.setdefault((_block_id(n), n.rsplit('.', 1)[-1], tuple(model_sd[n].shape)), []).append(n)
    for k, v in sd.items():
        if k not in used:
            groups_c.setdefault((_block_id(_strip(k)), k.rsplit('.', 1)[-1], tuple(v.shape)), []).append(k)
    n_shape = 0
    for sig, ms in groups_m.items():
        cs = groups_c.get(sig, [])
        if sig[0] is None or len(cs) != len(ms): continue
        for n, k in zip(ms, cs):
            mapped[n] = sd[k]; used.add(k); n_shape += 1
            if sig[0] == 0: print(f"   [{name}] by shape: {k} -> {n}")
    if n_shape: print(f"   [{name}] {n_shape} tensors matched by block/shape (block-0 mapping shown above)")

    if 'tok_emb.weight' not in mapped:
        for k, v in sd.items():
            if _strip(k).endswith(('lm_head.weight', 'head.weight', 'output.weight')) and tuple(v.shape) == tuple(model_sd['tok_emb.weight'].shape):
                mapped['tok_emb.weight'] = v; break
    for k, v in sd.items():
        if _strip(k).endswith(('lm_head.weight', 'head.weight', 'output.weight')) and tuple(v.shape) == tuple(backbone.tok_emb.weight.shape):
            backbone._lm_w = v.detach().clone(); break

    backbone.load_state_dict(mapped, strict=False)
    tot = sum(p.numel() for p in backbone.parameters())
    got = sum(p.numel() for n, p in backbone.named_parameters() if n in mapped)
    frac = got / max(1, tot)
    print(("✓ " if frac > 0.9 else "⚠⚠ ") + f"[{name}] loaded {len(mapped)} tensors | {frac*100:.1f}% of backbone parameters restored")
    leftover = [f"{k} {tuple(sd[k].shape)}" for k in sd if k not in used
                and not _strip(k).endswith(('lm_head.weight', 'head.weight', 'output.weight', 'cos_cached', 'sin_cached', 'pe', 'slopes', 'inv_freq', 'mask', 'bias_cache'))]
    if frac <= 0.9:
        missing = [n for n in param_names if n not in mapped]
        print(f"   WARNING: still missing e.g.: {missing[:8]}")
    if leftover:
        # Weights in the checkpoint that this architecture has no slot for (e.g. attention biases) are
        # silently dropped and would make the forward pass differ from Phase 1.
        print(f"   ⚠ {len(leftover)} checkpoint tensors were NOT used, e.g.: {leftover[:10]}")
    return frac

# ---------------------------------- LORMA -------------------------------------
class LORMALinear(nn.Module):
    """y = W0 x + s * (alpha/r) * B A x. B = 0 at init, so injecting the adapter does not change the
    pretrained model (the Stage-A head stays valid). A learnable magnitude s (init 1) rescales the update."""
    def __init__(self, base_layer: nn.Linear, rank: int = 16, alpha: float = 32.0, dropout: float = 0.05):
        super().__init__()
        self.base_layer, self.rank, self.scaling = base_layer, rank, alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        self.lora_A = nn.Parameter(torch.empty(rank, base_layer.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base_layer.out_features, rank))
        self.lora_s = nn.Parameter(torch.ones(1))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        for p in self.base_layer.parameters(): p.requires_grad = False
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        delta = (self.dropout(x) @ self.lora_A.t().to(x.dtype)) @ self.lora_B.t().to(x.dtype)
        return self.base_layer(x) + delta * (self.lora_s.to(x.dtype) * self.scaling)

def inject_lorma(model: nn.Module, r_attn: int = 16, r_ffn: int = 8, alpha: float = 32.0, dropout: float = 0.05,
                 attn_names=('q_proj', 'k_proj', 'v_proj', 'out_proj'), ffn_names=('gate_proj', 'up_proj', 'down_proj')):
    for p in model.parameters(): p.requires_grad = False
    count = 0
    for block in model.blocks:
        for name in attn_names:
            l = getattr(block.attn, name, None)
            if isinstance(l, nn.Linear):
                setattr(block.attn, name, LORMALinear(l, rank=r_attn, alpha=alpha, dropout=dropout)); count += 1
        for name in ffn_names:
            l = getattr(block.ffn, name, None)
            if isinstance(l, nn.Linear):
                setattr(block.ffn, name, LORMALinear(l, rank=r_ffn, alpha=alpha, dropout=dropout)); count += 1
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✓ Injected LORMA into {count} linear projections (r_attn={r_attn}, r_ffn={r_ffn}) | {n/1e6:.2f}M adapter params")

# -------------------------------- AdaLoRA -------------------------------------
class AdaLoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, r: int = 16, alpha: float = 32.0, dropout: float = 0.05):
        super().__init__()
        self.base_layer, self.r, self.scaling = base_layer, r, alpha / r
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        self.lora_P = nn.Parameter(torch.empty(base_layer.out_features, r))
        self.lora_E = nn.Parameter(torch.zeros(r))
        self.lora_Q = nn.Parameter(torch.empty(r, base_layer.in_features))
        nn.init.orthogonal_(self.lora_P); nn.init.orthogonal_(self.lora_Q)
        self.register_buffer('rank_mask', torch.ones(r))
        self.register_buffer('importance', torch.zeros(r))
        for p in self.base_layer.parameters(): p.requires_grad = False
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lam = (self.lora_E * self.rank_mask).to(x.dtype)
        h = (self.dropout(x) @ self.lora_Q.t().to(x.dtype)) * lam
        return self.base_layer(x) + (h @ self.lora_P.t().to(x.dtype)) * self.scaling
    def get_ortho_loss(self):
        P, Q = self.lora_P.float(), self.lora_Q.float()
        eye = torch.eye(self.r, device=P.device)
        return torch.norm(P.t() @ P - eye, p='fro') + torch.norm(Q @ Q.t() - eye, p='fro')

def inject_adalora(model: nn.Module, r: int = 16, alpha: float = 32.0):
    for p in model.parameters(): p.requires_grad = False
    count = 0
    for block in model.blocks:
        for name in ('q_proj', 'k_proj', 'v_proj', 'out_proj'):
            l = getattr(block.attn, name)
            if isinstance(l, nn.Linear):
                setattr(block.attn, name, AdaLoRALinear(l, r=r, alpha=alpha)); count += 1
        for name in ('fc1', 'fc2'):
            l = getattr(block.ffn, name)
            if isinstance(l, nn.Linear):
                setattr(block.ffn, name, AdaLoRALinear(l, r=r, alpha=alpha)); count += 1
    print(f"✓ Injected AdaLoRA into {count} linear projections.")

class AdaLoRAController:
    """Importance s_i = EMA(|Lambda_i * dL/dLambda_i|); the global top-b triplets are kept. The budget b
    follows a cubic schedule in the fraction of the time budget used (full rank -> final_ratio)."""
    def __init__(self, model, final_ratio=0.5, t_init=0.15, t_final=0.30, every=10, beta=0.85):
        self.mods = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
        self.full = sum(m.r for m in self.mods)
        self.target = int(self.full * final_ratio)
        self.t_init, self.t_final, self.every, self.beta, self.calls = t_init, t_final, every, beta, 0
    def budget(self, frac):
        if frac < self.t_init: return self.full
        if frac >= 1 - self.t_final: return self.target
        f = (frac - self.t_init) / max(1e-6, 1 - self.t_init - self.t_final)
        return int(self.target + (self.full - self.target) * (1 - f) ** 3)
    @torch.no_grad()
    def __call__(self, frac):
        for m in self.mods:
            if m.lora_E.grad is not None:
                s = (m.lora_E * m.lora_E.grad).abs()
                if torch.isfinite(s).all():
                    m.importance.mul_(self.beta).add_((1 - self.beta) * s)
        self.calls += 1
        if self.calls % self.every: return
        b = self.budget(frac)
        if b >= self.full: return
        scores = torch.cat([m.importance * m.rank_mask for m in self.mods])
        thr = torch.topk(scores, b).values[-1]
        for m in self.mods:
            m.rank_mask.copy_(((m.importance * m.rank_mask) >= thr).float() * m.rank_mask)
    def kept(self):
        return int(sum(m.rank_mask.sum().item() for m in self.mods))

ORTHO_GAMMA = 0.1

print("✓ Loader and adapters (LORMA, AdaLoRA) ready.")

In [ ]:
# 7. Backbone builders + diagnosis: which pre-tokeniser and which forward-pass variants reproduce Phase 1?
#    Measured by next-token perplexity on Dev text (Phase-1 Arch 4 reached 23.66 on its own validation set).
ARCH_SPECS = {
    4: dict(cls=lambda: ALiBiSwiGLULM(vocab_size=tokenizer.vocab_size), ckpt=lambda: ARCH4_CHECKPOINT),
    3: dict(cls=lambda: GQADecoderLM(vocab_size=tokenizer.vocab_size, max_seq_len=1024), ckpt=lambda: ARCH3_CHECKPOINT),
    2: dict(cls=lambda: RoPESwiGLUDecoderLM(vocab_size=tokenizer.vocab_size, max_seq_len=MAX_POS), ckpt=lambda: ARCH2_CHECKPOINT),
    1: dict(cls=lambda: StudentArch1LM(vocab_size=tokenizer.vocab_size, max_seq_len=MAX_POS), ckpt=lambda: ARCH1_CHECKPOINT),
}

def build_backbone(arch: int, name: str = None) -> nn.Module:
    bb = ARCH_SPECS[arch]['cls']()
    load_pretrained(bb, ARCH_SPECS[arch]['ckpt'](), name or f"Arch{arch}")
    bb = bb.to(device)
    disable_dropout(bb)
    return bb.eval()

def lm_weight(bb):
    W = getattr(bb, '_lm_w', None)
    if W is None:
        W = bb.lm_head.weight if hasattr(bb, 'lm_head') else bb.tok_emb.weight
    return W.to(device).float()

@torch.no_grad()
def lm_ppl(bb, ids: torch.Tensor, use_amp: bool = False, bs: int = 8) -> float:
    bb.eval(); W = lm_weight(bb); tot, n = 0.0, 0
    for s in range(0, ids.size(0), bs):
        x = ids[s:s + bs].to(device)
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp and USE_AMP):
            h = bb(x)
        h = h.float()
        if not torch.isfinite(h).all(): return float('inf')
        ce = F.cross_entropy((h @ W.t())[:, :-1].reshape(-1, W.size(0)), x[:, 1:].reshape(-1), reduction='sum')
        tot += ce.item(); n += x[:, 1:].numel()
    return math.exp(min(50.0, tot / max(1, n)))

def ppl_sample(mode: str, n: int = 24, seed: int = 0) -> torch.Tensor:
    """n Dev chunks of CHUNK_LEN tokens, each taken from the middle of a different document."""
    docs = random.Random(seed).sample(dev_records, min(400, len(dev_records)))
    rows = []
    for r in docs:
        words = pretokenize(r['text'], mode)
        if len(words) < CHUNK_LEN * 2: continue
        mid = len(words) // 2
        ids = tokenizer.encode_words(words[mid:mid + CHUNK_LEN])[:CHUNK_LEN]
        if len(ids) == CHUNK_LEN: rows.append(ids)
        if len(rows) >= n: break
    return torch.tensor(rows, dtype=torch.long)

DIAG = {'vocab': VOCAB_INFO}
_samples = {m: ppl_sample(m) for m in PREPROC_MODES}

# --- (a) pre-tokeniser, judged on the two simplest backbones -------------------
if PREPROC == 'auto':
    table = {}
    for arch in (4, 1):
        bb = build_backbone(arch, f"Arch{arch} (pre-tokeniser check)")
        table[arch] = {m: lm_ppl(bb, _samples[m]) for m in PREPROC_MODES}
        del bb; free_mem()
    print(f"\n{'pre-tokeniser':>14} | " + " | ".join(f"Arch{a} ppl" for a in table))
    for m in PREPROC_MODES:
        print(f"{m:>14} | " + " | ".join(f"{table[a][m]:9.1f}" for a in table))
    PREPROC = min(PREPROC_MODES, key=lambda m: sum(math.log(table[a][m]) for a in table))
    DIAG['preproc_table'] = {str(a): v for a, v in table.items()}
    print(f"✓ Chosen pre-tokeniser: '{PREPROC}' (vocabulary hint was '{VOCAB_INFO['hint']}')")
DIAG['preproc'] = PREPROC
PPL_IDS = _samples[PREPROC]

# --- (b) forward-pass variants per architecture, on the chosen tokens --------
VARIANT_GRID = {
    3: [{'kv': kv, 'act': a} for kv in ('interleave', 'repeat') for a in ('gelu', 'gelu_tanh', 'relu')],
    2: [{'rope': r, 'qkn': q, 'swap': s} for r in ('half', 'pairs') for q in ('before', 'after') for s in (False, True)],
    1: [{'act': a} for a in ('gelu', 'gelu_tanh', 'relu')],
}
DIAG['ppl'] = {}
for arch in (4, 3, 2, 1):
    if not ARCH_SPECS[arch]['ckpt'](): continue
    bb = build_backbone(arch, f"Arch{arch} (variant check)")
    if arch in VARIANT_GRID:
        res = []
        for opt in VARIANT_GRID[arch]:
            ARCH_OPTS[arch] = dict(opt); res.append((lm_ppl(bb, PPL_IDS), opt))
        res.sort(key=lambda r: r[0])
        ARCH_OPTS[arch] = dict(res[0][1])
        print(f"Arch{arch} variants: " + " | ".join(f"{o} -> {p:.1f}" for p, o in res[:4]) + (" ..." if len(res) > 4 else ""))
    p = lm_ppl(bb, PPL_IDS)
    DIAG['ppl'][arch] = p
    print(f"✓ Arch{arch}: perplexity {p:.1f} with {ARCH_OPTS.get(arch, '(fixed)')}")
    del bb; free_mem()

print("\nSUMMARY  pre-tokeniser:", PREPROC, "| perplexity per arch:", {a: round(p, 1) for a, p in DIAG['ppl'].items()})
if max(DIAG['ppl'].values(), default=0) > 150:
    print("⚠ Some perplexities are still far above Phase 1. If you have the Phase-1 notebook, compare its text\n"
          "  cleaning (e.g. unicode normalisation, removal of line numbers) and its model classes with cells 3 and 5.")
DIAG['arch_opts'] = {str(k): v for k, v in ARCH_OPTS.items()}
with open(os.path.join(OUT_DIR, "diagnostics.json"), 'w') as f: json.dump(DIAG, f, indent=2, default=str)
del _samples; free_mem()

In [10]:
# Re-threshold Run 1 using the committed run's saved predictions (no retraining)
import numpy as np, glob, json, zipfile, os
from sklearn.metrics import f1_score

KEY = 'Run1_Arch4_LORMA'
def _find(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True) +
                  glob.glob(f'/kaggle/working/**/{name}', recursive=True))
    assert hits, f'{name} not found - attach the committed notebook version as an input'
    print('using', hits[0]); return hits[0]

pd_ = np.load(_find(f'{KEY}_dev.npy'))[:, 1]
pt_ = np.load(_find(f'{KEY}_test.npy'))[:, 1]
meta = json.load(open(_find(f'{KEY}_meta.json')))
y = np.asarray(dev_labels)
assert len(pd_) == len(y) and len(pt_) == len(test_ids), 'Dev/Test sizes differ from the saved run'

prior = y.mean()
tau_old = meta['threshold']
tau_new = np.quantile(pt_, 1 - prior)          # Test threshold giving a class-1 rate = Dev prior
tau_dev = np.quantile(pd_, 1 - prior)          # same rule applied on Dev, to check it there

f1 = lambda p, t: f1_score(y, (p >= t).astype(int), average='macro')
print(f"Dev class-1 rate (truth):     {prior:.3f}")
print(f"Old tau {tau_old:.3f}: Dev F1 {f1(pd_, tau_old):.4f} | Dev pred-rate {(pd_>=tau_old).mean():.3f} | Test pred-rate {(pt_>=tau_old).mean():.3f}")
print(f"Prior-matched rule:  Dev F1 {f1(pd_, tau_dev):.4f} | Test tau {tau_new:.3f} -> Test pred-rate {(pt_>=tau_new).mean():.3f}")

preds = (pt_ >= tau_new).astype(int)
os.makedirs('/kaggle/working/rethresh', exist_ok=True)
jp = '/kaggle/working/rethresh/prediction.json'
json.dump([{"id": i, "pred": int(p)} for i, p in zip(test_ids, preds)], open(jp, 'w'), indent=2)
with zipfile.ZipFile('/kaggle/working/submission_run1_prior_matched.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(jp, arcname='prediction.json')
print(f"✓ submission_run1_prior_matched.zip | class 1: {preds.sum()} / {len(preds)}")

using /kaggle/input/notebooks/brindafadadu/phase4-allrun/preds/Run1_Arch4_LORMA_dev.npy
using /kaggle/input/notebooks/brindafadadu/phase4-allrun/preds/Run1_Arch4_LORMA_test.npy
using /kaggle/input/notebooks/brindafadadu/phase4-allrun/preds/Run1_Arch4_LORMA_meta.json
Dev class-1 rate (truth):     0.500
Old tau 0.448: Dev F1 0.6465 | Dev pred-rate 0.450 | Test pred-rate 0.434
Prior-matched rule:  Dev F1 0.6398 | Test tau 0.402 -> Test pred-rate 0.500
✓ submission_run1_prior_matched.zip | class 1: 759 / 1517


In [11]:
# Can any combination of saved models rank Dev better than Run 1 alone? (CPU, uses committed preds)
import numpy as np, glob, os, json, zipfile
from sklearn.metrics import f1_score, roc_auc_score
from scipy.stats import rankdata

D = os.path.dirname(glob.glob('/kaggle/input/**/preds/Run1_Arch4_LORMA_dev.npy', recursive=True)[0])
y = np.asarray(dev_labels)
M = {}
for p in sorted(glob.glob(f'{D}/*_dev.npy')):
    k, tp = os.path.basename(p)[:-8], p.replace('_dev.npy', '_test.npy')
    if os.path.exists(tp):
        d, t = np.load(p)[:, 1], np.load(tp)[:, 1]
        if len(d) == len(y) and len(t) == len(test_ids): M[k] = (d, t)

pct = lambda a: (rankdata(a) - 0.5) / len(a)          # rank-normalise so models are on one scale
def best_rate(s):                                     # best Dev Macro-F1 over predicted-positive rates
    return max((f1_score(y, (s >= np.quantile(s, 1 - r)).astype(int), average='macro'), r)
               for r in np.arange(0.30, 0.71, 0.01))

print(f"Dev size {len(y)} | class-1 rate {y.mean():.3f}\n\nMembers:")
for k, (d, _) in sorted(M.items(), key=lambda kv: -roc_auc_score(y, kv[1][0])):
    f, r = best_rate(d); print(f"  {k:28s} AUC {roc_auc_score(y, d):.4f} | F1 {f:.4f} @rate {r:.2f}")

R1 = 'Run1_Arch4_LORMA'
base_auc = roc_auc_score(y, M[R1][0])
cands = {}
for k in M:
    if k == R1: continue
    for w in (0.5, 0.67, 0.8):
        cands[f'{R1}*{w}+{k}'] = {R1: w, k: 1 - w}
top = sorted(M, key=lambda k: -roc_auc_score(y, M[k][0]))
for n in (3, 4, 5):
    if len(top) >= n: cands[f'avg-top{n}'] = {k: 1 / n for k in top[:n]}

res = []
for name, w in cands.items():
    sd = sum(v * pct(M[k][0]) for k, v in w.items())
    res.append((roc_auc_score(y, sd), name, w))
res.sort(reverse=True, key=lambda r: r[0])
print(f"\nRun 1 alone: AUC {base_auc:.4f}\nBest blends:")
for a, name, _ in res[:6]: print(f"  {name:55s} AUC {a:.4f} ({a - base_auc:+.4f})")

a, name, w = res[0]
if a < base_auc + 0.01:
    print("\n→ No blend beats Run 1 by ≥0.01 Dev AUC. Keep the 0.6377 submission.")
else:
    sd = sum(v * pct(M[k][0]) for k, v in w.items()); st = sum(v * pct(M[k][1]) for k, v in w.items())
    f, r = best_rate(sd)
    preds = (st >= np.quantile(st, 1 - r)).astype(int)
    os.makedirs('/kaggle/working/blend', exist_ok=True); jp = '/kaggle/working/blend/prediction.json'
    json.dump([{"id": i, "pred": int(p)} for i, p in zip(test_ids, preds)], open(jp, 'w'), indent=2)
    with zipfile.ZipFile('/kaggle/working/submission_blend.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname='prediction.json')
    print(f"\n→ {name}: Dev AUC {a:.4f}, Dev F1 {f:.4f} @rate {r:.2f} | wrote submission_blend.zip ({preds.sum()} class 1)")

Dev size 1988 | class-1 rate 0.500

Members:
  Run1_Arch4_LORMA             AUC 0.6885 | F1 0.6465 @rate 0.45
  Run1_Arch4_LORMA_probe       AUC 0.6487 | F1 0.6296 @rate 0.48
  Fusion_FrozenLR              AUC 0.6403 | F1 0.6116 @rate 0.51
  Run3_Arch2_Layerwise_probe   AUC 0.6374 | F1 0.6035 @rate 0.41
  Run4_KD_Arch4_to_Arch1       AUC 0.6339 | F1 0.6011 @rate 0.33
  Run4_KD_Arch4_to_Arch1_probe AUC 0.6277 | F1 0.5952 @rate 0.53
  Run2_Arch3_AdaLoRA           AUC 0.6256 | F1 0.5994 @rate 0.52
  Run3_Arch2_Layerwise         AUC 0.6229 | F1 0.5900 @rate 0.42
  Run2_Arch3_AdaLoRA_probe     AUC 0.6092 | F1 0.5975 @rate 0.49
  Run1_Arch4_LORMA_frozen      AUC 0.5169 | F1 0.5211 @rate 0.42
  Run3_Arch2_Layerwise_frozen  AUC 0.5155 | F1 0.5151 @rate 0.50
  Run2_Arch3_AdaLoRA_frozen    AUC 0.5027 | F1 0.5117 @rate 0.52
  Run4_KD_Arch4_to_Arch1_frozen AUC 0.4723 | F1 0.4926 @rate 0.55

Run 1 alone: AUC 0.6885
Best blends:
  Run1_Arch4_LORMA*0.8+Run4_KD_Arch4_to_Arch1_probe       AUC 0.6946 (+

In [12]:
# Ensemble: Run 1 (dominant) + Arch3 AdaLoRA + Arch1 KD probe, rank-averaged
import numpy as np, glob, os, json, zipfile
from sklearn.metrics import f1_score, roc_auc_score
from scipy.stats import rankdata

D = os.path.dirname(glob.glob('/kaggle/input/**/preds/Run1_Arch4_LORMA_dev.npy', recursive=True)[0])
W = {'Run1_Arch4_LORMA': 0.70, 'Run2_Arch3_AdaLoRA': 0.15, 'Run4_KD_Arch4_to_Arch1_probe': 0.15}
y = np.asarray(dev_labels)
pct = lambda a: (rankdata(a) - 0.5) / len(a)
ld = lambda k, s: np.load(f'{D}/{k}_{s}.npy')[:, 1]

sd = sum(w * pct(ld(k, 'dev'))  for k, w in W.items())
st = sum(w * pct(ld(k, 'test')) for k, w in W.items())

f1r = lambda s, r: f1_score(y, (s >= np.quantile(s, 1 - r)).astype(int), average='macro')
rates = np.arange(0.38, 0.53, 0.01)
r = max(rates, key=lambda r: f1r(sd, r))
r1 = pct(ld('Run1_Arch4_LORMA', 'dev'))
print(f"Run 1 alone : Dev AUC {roc_auc_score(y, r1):.4f} | F1 {max(f1r(r1, q) for q in rates):.4f}")
print(f"Ensemble    : Dev AUC {roc_auc_score(y, sd):.4f} | F1 {f1r(sd, r):.4f} @rate {r:.2f}")

preds = (st >= np.quantile(st, 1 - r)).astype(int)
os.makedirs('/kaggle/working/ens', exist_ok=True); jp = '/kaggle/working/ens/prediction.json'
json.dump([{"id": i, "pred": int(p)} for i, p in zip(test_ids, preds)], open(jp, 'w'), indent=2)
with zipfile.ZipFile('/kaggle/working/submission_ensemble_v2.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(jp, arcname='prediction.json')
print(f"✓ submission_ensemble_v2.zip | class 1: {preds.sum()} / {len(preds)} ({preds.mean():.1%})")

Run 1 alone : Dev AUC 0.6885 | F1 0.6465
Ensemble    : Dev AUC 0.6970 | F1 0.6568 @rate 0.39
✓ submission_ensemble_v2.zip | class 1: 592 / 1517 (39.0%)


In [9]:
# 8. Head + tail chunking and a compact token store (uses the pre-tokeniser chosen in cell 7)
#    Every document becomes the SAME set of chunks at train and eval time:
#    HEAD_CHUNKS x 512 tokens from the start + TAIL_CHUNKS x 512 tokens ending at the last token.
#    Long documents are only tokenised at their two ends (each word gives >= 1 token), which
#    makes pre-tokenisation several times faster than encoding the full judgment.
K_DOC = HEAD_CHUNKS + TAIL_CHUNKS

def _enc_words(words):
    return tokenizer.encode_words(words)

def split_tokens(toks: List[int]) -> List[List[int]]:
    L, H, T = CHUNK_LEN, HEAD_CHUNKS, TAIL_CHUNKS
    n = len(toks)
    if n == 0:
        return [[SEP_ID]]
    if n <= L:
        return [toks]
    if n <= K_DOC * L:
        starts = list(range(0, n - L + 1, L))
        if starts[-1] != n - L:
            starts.append(n - L)                      # last chunk ends exactly at the document end
        if len(starts) > K_DOC:
            starts = starts[:H] + starts[len(starts) - T:]
        return [toks[s:s + L] for s in starts]
    starts = [i * L for i in range(H)] + [n - (T - j) * L for j in range(T)]
    return [toks[s:s + L] for s in starts]

def doc_to_chunks(text: str, words: Optional[List[str]] = None) -> List[List[int]]:
    L, H, T = CHUNK_LEN, HEAD_CHUNKS, TAIL_CHUNKS
    words = pretokenize(text, PREPROC) if words is None else words
    if len(words) > K_DOC * L:                       # tokens >= words, so head and tail cannot overlap
        head = _enc_words(words[:H * L])[:H * L] if H > 0 else []
        tail = _enc_words(words[-T * L:])[-T * L:]
        return [head[i * L:(i + 1) * L] for i in range(H)] + [tail[j * L:(j + 1) * L] for j in range(T)]
    return split_tokens(_enc_words(words))

class TokStore:
    """All chunks of a split in one int16 matrix; doc i owns rows ptr[i]:ptr[i+1]."""
    def __init__(self, records=None, name=''):
        if records is None:
            return
        t0 = time.time()
        rows, lens, ptr, n_unk, n_tok, n_words = [], [], [0], 0, 0, []
        for r in records:
            words = pretokenize(r['text'], PREPROC)
            ch = doc_to_chunks(r['text'], words)
            for c in ch:
                rows.append(c); lens.append(len(c))
                n_unk += sum(1 for x in c if x == UNK_ID); n_tok += len(c)
            ptr.append(ptr[-1] + len(ch)); n_words.append(len(words))
        dt = np.int16 if tokenizer.vocab_size < 32000 else np.int32
        self.flat = np.full((len(rows), CHUNK_LEN), PAD_ID, dtype=dt)
        for j, c in enumerate(rows):
            self.flat[j, :len(c)] = c
        self.lens = np.array(lens, dtype=np.int64)
        self.ptr = np.array(ptr, dtype=np.int64)
        self.ids = [r['id'] for r in records]
        self.labels = np.array([(-1 if r['label'] is None else int(r['label'])) for r in records], dtype=np.int64)
        nw = np.array(n_words)
        print(f"✓ [{name}] {len(records)} docs -> {len(rows)} chunks in {time.time()-t0:.0f}s | "
              f"words/doc median={int(np.median(nw))}, p90={int(np.percentile(nw, 90))} | "
              f"chunks/doc={len(rows)/max(1,len(records)):.2f} | <unk> rate={n_unk/max(1,n_tok)*100:.2f}%")
        if n_unk / max(1, n_tok) > 0.05:
            print("  ⚠⚠ more than 5% <unk> tokens: the tokenizer file does not match this text -> features will be poor.")

    def __len__(self):
        return len(self.ids)

    def get(self, i: int):
        a, b = self.ptr[i], self.ptr[i + 1]
        ids = torch.from_numpy(self.flat[a:b].astype(np.int64))
        mask = torch.arange(CHUNK_LEN)[None, :] < torch.from_numpy(self.lens[a:b])[:, None]
        return ids, mask

    def subset(self, idx) -> 'TokStore':
        s = TokStore()
        rows = np.concatenate([np.arange(self.ptr[i], self.ptr[i + 1]) for i in idx])
        s.flat, s.lens = self.flat[rows], self.lens[rows]
        s.ptr = np.concatenate([[0], np.cumsum([self.ptr[i + 1] - self.ptr[i] for i in idx])]).astype(np.int64)
        s.ids = [self.ids[i] for i in idx]; s.labels = self.labels[np.asarray(idx)]
        return s

def build_store(records, name):
    path = os.path.join(WORK_DIR, f"tok_{name}_{len(records)}_{CHUNK_LEN}_{HEAD_CHUNKS}_{TAIL_CHUNKS}_{PREPROC}.pkl")
    if os.path.exists(path):
        with open(path, 'rb') as f:
            s = pickle.load(f)
        print(f"✓ [{name}] loaded token cache ({len(s)} docs)")
        return s
    s = TokStore(records, name)
    with open(path, 'wb') as f:
        pickle.dump(s, f)
    return s

train_store = build_store(train_records, 'train')
dev_store   = build_store(dev_records, 'dev')
test_store  = build_store(test_records, 'test')
dev_labels  = dev_store.labels
test_ids    = test_store.ids
print(f"Dev label balance: {np.bincount(dev_labels, minlength=2)}")

✓ [train] 37387 docs -> 128249 chunks in 38s | words/doc median=2192, p90=6575 | chunks/doc=3.43 | <unk> rate=0.00%
✓ [dev] 1988 docs -> 7662 chunks in 2s | words/doc median=2821, p90=6933 | chunks/doc=3.85 | <unk> rate=0.00%
✓ [test] 1517 docs -> 5842 chunks in 2s | words/doc median=2940, p90=7302 | chunks/doc=3.85 | <unk> rate=0.00%
Dev label balance: [994 994]


In [ ]:
# 9. Shared engine
#    - backbone sanity check (perplexity, fp16 safety)
#    - FeatureEncoder: per chunk, mean-pool the final layer AND blocks POOL_LAYERS (RMS-normalised)
#    - document heads; every head starts with a FIXED per-dimension standardisation (train mean / std)
#    - Stage A: head on cached frozen features (optionally with teacher soft labels)
#    - Stage B: PEFT fine-tuning, several documents per forward pass, OOM-safe
#    - linear probe on GPU (Check 1, and the fusion member of Run 5), threshold calibration, persistence
RUN_PREDICTIONS_DEV, RUN_PREDICTIONS_TEST, RUN_METRICS_SUMMARY = {}, {}, {}

@torch.no_grad()
def backbone_sanity(backbone: nn.Module, name: str):
    global AMP_ON
    p32 = lm_ppl(backbone, PPL_IDS, use_amp=False)
    msg = f"[{name}] sanity: Dev next-token perplexity (fp32) = {p32:.1f}"
    if USE_AMP:
        pam = lm_ppl(backbone, PPL_IDS, use_amp=True)
        msg += f" | with {str(AMP_DTYPE).split('.')[-1]} autocast = {pam:.1f}"
        if not math.isfinite(pam) or abs(math.log(max(pam, 1e-6)) - math.log(max(p32, 1e-6))) > 0.15:
            AMP_ON = False
            msg += "  -> mixed precision is unsafe for this backbone, running it in fp32"
    print(msg)
    return p32

# ---------------------------- chunk feature encoder ---------------------------
def pool_multi(last: torch.Tensor, mids, m: torch.Tensor) -> torch.Tensor:
    mf = m.unsqueeze(-1).float()
    n = mf.sum(1).clamp(min=1.0)
    outs = [(last.float() * mf).sum(1) / n]
    for h in mids:
        h = h.float()
        h = h * torch.rsqrt(h.pow(2).mean(-1, keepdim=True) + 1e-6)      # un-normalised residual stream
        outs.append((h * mf).sum(1) / n)
    return torch.cat(outs, dim=-1)

class FeatureEncoder(nn.Module):
    """backbone -> (n_chunks, (len(POOL_LAYERS)+1) * D) chunk vectors."""
    def __init__(self, backbone: nn.Module, layers=POOL_LAYERS):
        super().__init__()
        self.backbone = backbone
        nb = len(backbone.blocks)
        self.layers = [l for l in layers if 0 <= l < nb - 1]
        self.out_dim = (len(self.layers) + 1) * backbone.tok_emb.embedding_dim
        self._buf = {}
        for li in self.layers:
            backbone.blocks[li].register_forward_hook(partial(self._grab, li))
    def _grab(self, li, module, inp, out):
        self._buf[li] = out[0] if isinstance(out, tuple) else out
    def _run(self, ids):
        self._buf = {}
        last = self.backbone(ids)
        mids = tuple(self._buf[li] for li in self.layers)
        self._buf = {}
        return (last,) + mids
    def forward(self, ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        bs = ENC_BS if self.training else EXTRACT_BS
        outs = []
        for s in range(0, ids.size(0), bs):
            x, m = ids[s:s + bs], mask[s:s + bs]
            if self.training and GRAD_CKPT and torch.is_grad_enabled():
                hs = checkpoint(self._run, x, use_reentrant=False)
            else:
                hs = self._run(x)
            outs.append(pool_multi(hs[0], hs[1:], m))
        return torch.cat(outs, 0)

def to_padded(flat_feats: torch.Tensor, store: TokStore):
    N, Fd = len(store), flat_feats.size(-1)
    X = torch.zeros(N, K_DOC, Fd, dtype=flat_feats.dtype); M = torch.zeros(N, K_DOC, dtype=torch.bool)
    for i in range(N):
        a, b = int(store.ptr[i]), int(store.ptr[i + 1])
        X[i, :b - a] = flat_feats[a:b]; M[i, :b - a] = True
    return X, M

@torch.no_grad()
def extract_features(encoder: FeatureEncoder, store: TokStore, desc: str = '', cache_key: Optional[str] = None):
    path = os.path.join(FEAT_DIR, f"{cache_key}.pt") if (cache_key and CACHE_FEATS_TO_DISK) else None
    if path and os.path.exists(path):
        d = torch.load(path); print(f"  loaded cached features {cache_key}"); return d['X'], d['M']
    was_training = encoder.training
    encoder.eval()
    Nc = store.flat.shape[0]
    out = torch.empty(Nc, encoder.out_dim, dtype=torch.float16)
    t0, ar = time.time(), torch.arange(CHUNK_LEN, device=device)[None]
    step, every = EXTRACT_BS, max(1, Nc // 5); next_report = every
    for s in range(0, Nc, step):
        ids = torch.from_numpy(store.flat[s:s + step].astype(np.int64)).to(device, non_blocking=True)
        mask = ar < torch.from_numpy(store.lens[s:s + step]).to(device)[:, None]
        with amp_ctx():
            f = encoder(ids, mask)
        out[s:s + len(ids)] = f.float().clamp(-6e4, 6e4).cpu().half()
        done = s + len(ids)
        if desc and Nc >= 2000 and done >= next_report and done < Nc:
            el = time.time() - t0; next_report += every
            print(f"    [{desc}] {done}/{Nc} chunks | {el/60:.1f} min elapsed, ~{el/done*(Nc-done)/60:.1f} min left")
    if desc:
        print(f"  ✓ [{desc}] features for {Nc} chunks in {(time.time()-t0)/60:.1f} min")
    if not torch.isfinite(out.float()).all():
        print(f"  ⚠⚠ [{desc}] non-finite features found")
    encoder.train(was_training)
    X, M = to_padded(out, store)
    if path: torch.save({'X': X, 'M': M}, path)
    return X, M

def feat_stats(X: torch.Tensor, M: torch.Tensor):
    """Per-dimension mean / std over all real chunks (float64 accumulation)."""
    Xf = X[M]
    s = torch.zeros(Xf.size(-1), dtype=torch.float64); s2 = torch.zeros_like(s)
    for i in range(0, Xf.size(0), 8192):
        b = Xf[i:i + 8192].double(); s += b.sum(0); s2 += (b * b).sum(0)
    n = max(1, Xf.size(0)); mu = s / n
    sd = (s2 / n - mu * mu).clamp(min=0).sqrt()
    sd = sd.clamp(min=float(sd.median()) * 1e-2 + 1e-8)
    return mu.float(), sd.float()

def doc_views(X: torch.Tensor, M: torch.Tensor, bs: int = 4096) -> np.ndarray:
    """Doc-level vector = [mean over chunks, last chunk] (frozen features for probes and fusion)."""
    out = []
    for s in range(0, X.size(0), bs):
        x, m = X[s:s + bs].float(), M[s:s + bs]
        mf = m.unsqueeze(-1).float()
        mean = (x * mf).sum(1) / mf.sum(1).clamp(min=1)
        last = x[torch.arange(x.size(0)), (m.sum(1) - 1).clamp(min=0)]
        out.append(torch.cat([mean, last], -1).half().numpy())
    return np.concatenate(out, 0)

# ---------------------------- document heads ---------------------------------
class Standardize(nn.Module):
    """Fixed per-dimension standardisation. LayerNorm alone lets a few huge residual-stream
    dimensions squash all others; this is why the v3 heads of Runs 2-4 never left the class prior."""
    def __init__(self, dim):
        super().__init__()
        self.register_buffer('mu', torch.zeros(dim)); self.register_buffer('inv_sd', torch.ones(dim))
    def set_stats(self, mu, sd):
        self.mu.copy_(mu.float().to(self.mu.device)); self.inv_sd.copy_((1.0 / sd.float()).to(self.inv_sd.device))
    def forward(self, x):
        return ((x.float() - self.mu) * self.inv_sd).clamp(-10, 10)

def set_head_stats(head: nn.Module, mu, sd):
    for m in head.modules():
        if isinstance(m, Standardize): m.set_stats(mu, sd)

def _inp_block(in_dim, d, drop):
    return nn.Sequential(Standardize(in_dim), nn.Dropout(drop), nn.Linear(in_dim, d), nn.LayerNorm(d), nn.GELU())

def _masked_softmax_pool(h, m, scorer):
    s = scorer(h).squeeze(-1).masked_fill(~m, -1e4)
    return (h * torch.softmax(s, dim=-1).unsqueeze(-1)).sum(1)

def _masked_mean(h, m):
    mf = m.unsqueeze(-1).float()
    return (h * mf).sum(1) / mf.sum(1).clamp(min=1.0)

def _masked_max(h, m):
    return h.masked_fill(~m.unsqueeze(-1), -1e4).max(1).values

class BiLSTMDocHead(nn.Module):
    """Run 1 / Run 4: chunk vectors -> BiLSTM over chunk order -> attention + max pooling -> linear."""
    def __init__(self, in_dim, d=384, hidden=256, drop=HEAD_DROP):
        super().__init__()
        self.inp = _inp_block(in_dim, d, drop)
        self.lstm = nn.LSTM(d, hidden, batch_first=True, bidirectional=True)
        self.attn = nn.Sequential(nn.Linear(2 * hidden, 64), nn.Tanh(), nn.Linear(64, 1, bias=False))
        self.norm = nn.LayerNorm(4 * hidden); self.drop = nn.Dropout(drop)
        self.out = nn.Linear(4 * hidden, 2)
    def forward(self, x, m):
        h = self.inp(x.float())
        lens = m.sum(1).clamp(min=1).cpu()
        packed = pack_padded_sequence(h, lens, batch_first=True, enforce_sorted=False)
        o, _ = self.lstm(packed)
        o, _ = pad_packed_sequence(o, batch_first=True, total_length=x.size(1))
        feat = self.norm(torch.cat([_masked_softmax_pool(o, m, self.attn), _masked_max(o, m)], dim=-1))
        return self.out(self.drop(feat)), feat

class GatedAttnDocHead(nn.Module):
    """Run 2: gated (sigmoid x tanh) chunk units + chunk-position embeddings + self-attention across chunks."""
    def __init__(self, in_dim, d=384, drop=HEAD_DROP, n_heads=4):
        super().__init__()
        self.inp = _inp_block(in_dim, d, drop)
        self.g, self.f = nn.Linear(d, d), nn.Linear(d, d)
        self.pos = nn.Embedding(64, d)
        self.mha = nn.MultiheadAttention(d, n_heads, dropout=drop, batch_first=True)
        self.ln = nn.LayerNorm(d); self.score = nn.Linear(d, 1)
        self.norm = nn.LayerNorm(2 * d); self.drop = nn.Dropout(drop)
        self.out = nn.Linear(2 * d, 2)
    def forward(self, x, m):
        h = self.inp(x.float())
        h = torch.sigmoid(self.g(h)) * torch.tanh(self.f(h))
        h = h + self.pos(torch.arange(h.size(1), device=h.device).clamp(max=63))[None]
        a, _ = self.mha(h, h, h, key_padding_mask=~m, need_weights=False)
        h = self.ln(h + a)
        feat = self.norm(torch.cat([_masked_softmax_pool(h, m, self.score), _masked_mean(h, m)], dim=-1))
        return self.out(self.drop(feat)), feat

class TemporalConvDocHead(nn.Module):
    """Run 3: 1-D convolution over the chunk sequence -> masked max + mean pooling."""
    def __init__(self, in_dim, d=384, hidden=256, drop=HEAD_DROP):
        super().__init__()
        self.inp = _inp_block(in_dim, d, drop)
        self.conv = nn.Conv1d(d, hidden, kernel_size=3, padding=1)
        self.norm = nn.LayerNorm(2 * hidden); self.drop = nn.Dropout(drop)
        self.out = nn.Linear(2 * hidden, 2)
    def forward(self, x, m):
        h = self.inp(x.float()) * m.unsqueeze(-1).float()
        c = F.gelu(self.conv(h.transpose(1, 2))).transpose(1, 2)
        feat = self.norm(torch.cat([_masked_max(c, m), _masked_mean(c, m)], dim=-1))
        return self.out(self.drop(feat)), feat

class DocModel(nn.Module):
    """Backbone (with adapters) + document head; several documents per forward pass in Stage B."""
    def __init__(self, encoder: FeatureEncoder, head: nn.Module):
        super().__init__()
        self.encoder, self.head = encoder, head
    def forward(self, ids, mask, counts, return_feat=False):
        f = self.encoder(ids, mask)                                         # (sum(counts), F)
        X = torch.nn.utils.rnn.pad_sequence(list(torch.split(f.float(), list(counts))), batch_first=True)
        M = torch.arange(X.size(1), device=X.device)[None] < torch.as_tensor(counts, device=X.device)[:, None]
        with torch.autocast(device_type=device.type, enabled=False):
            logits, feat = self.head(X, M)
        return (logits, feat) if return_feat else logits

@torch.no_grad()
def head_predict(head: nn.Module, X, M, bs=256):
    was = head.training; head.eval()
    L, Fs = [], []
    for s in range(0, X.size(0), bs):
        lg, ft = head(X[s:s + bs].to(device).float(), M[s:s + bs].to(device))
        L.append(lg.float().cpu()); Fs.append(ft.float().cpu())
    head.train(was)
    return torch.cat(L), torch.cat(Fs)

def predict_store(model: DocModel, store: TokStore, desc=''):
    X, M = extract_features(model.encoder, store, desc)
    return head_predict(model.head, X, M)

def softmax_np(logits: torch.Tensor) -> np.ndarray:
    return F.softmax(logits.float(), dim=-1).numpy()

# ---------------------------- threshold calibration ---------------------------
def calibrate_threshold(probs: np.ndarray, labels: np.ndarray) -> Tuple[float, float, float]:
    """Candidate thresholds are QUANTILES of the scores (5%..95%): the chosen one always predicts both classes."""
    p1 = probs[:, 1]
    ts = np.unique(np.quantile(p1, np.linspace(0.05, 0.95, 91)))
    f1s = np.array([f1_score(labels, (p1 >= t).astype(int), average='macro', zero_division=0) for t in ts])
    k = min(5, len(f1s))
    sm = np.convolve(np.pad(f1s, (k // 2, k // 2), mode='edge'), np.ones(k) / k, mode='valid')[:len(f1s)]
    i = int(np.argmax(sm))
    default_f1 = f1_score(labels, (p1 >= 0.5).astype(int), average='macro', zero_division=0)
    return float(ts[i]), float(f1s[i]), float(default_f1)

def safe_auc(y, p):
    try: return float(roc_auc_score(y, p))
    except Exception: return float('nan')

def dev_score(f1, auc):
    return f1 + 0.1 * (0.0 if math.isnan(auc) else auc)

# ---------------------------- linear probe on GPU -----------------------------
def fit_linear(Xtr: np.ndarray, ytr: np.ndarray, Xdv: np.ndarray, ydv: np.ndarray, Xte: Optional[np.ndarray] = None,
               name: str = '', l2s=(1e-4, 1e-3, 1e-2), epochs: int = 25, bs: int = 2048, lr: float = 1e-3, verbose=True):
    """Standardised logistic regression, minibatch Adam on GPU, L2 + early stopping chosen on Dev.
    X* are (N, D) float16 numpy arrays. Returns (dev_probs, test_probs or None, info)."""
    t0 = time.time(); D = Xtr.shape[1]
    s = np.zeros(D, np.float64); s2 = np.zeros(D, np.float64)
    for i in range(0, len(Xtr), 4096):
        b = Xtr[i:i + 4096].astype(np.float64); s += b.sum(0); s2 += (b * b).sum(0)
    mu = s / len(Xtr); sd = np.sqrt(np.maximum(s2 / len(Xtr) - mu * mu, 0))
    sd = np.maximum(sd, np.median(sd) * 1e-2 + 1e-8)
    mu_t, isd_t = torch.tensor(mu, dtype=torch.float32, device=device), torch.tensor(1 / sd, dtype=torch.float32, device=device)
    def prep(x): return ((torch.from_numpy(np.ascontiguousarray(x)).to(device).float() - mu_t) * isd_t).clamp(-10, 10)
    @torch.no_grad()
    def predict(lin, X):
        return np.concatenate([torch.sigmoid(lin(prep(X[i:i + 8192])).squeeze(-1)).cpu().numpy() for i in range(0, len(X), 8192)])
    pos_w = torch.tensor(float((ytr == 0).sum()) / max(1, int((ytr == 1).sum())), dtype=torch.float32, device=device)
    best = (-1.0, None, None)
    for l2 in l2s:
        torch.manual_seed(SEED)
        lin = nn.Linear(D, 1).to(device); nn.init.zeros_(lin.weight); nn.init.zeros_(lin.bias)
        opt = torch.optim.Adam(lin.parameters(), lr=lr)
        yt = torch.as_tensor(ytr, dtype=torch.float32)
        for ep in range(epochs):
            perm = np.random.RandomState(SEED + ep).permutation(len(Xtr))
            for i in range(0, len(perm), bs):
                idx = np.sort(perm[i:i + bs])
                logit = lin(prep(Xtr[idx])).squeeze(-1)
                loss = F.binary_cross_entropy_with_logits(logit, yt[idx].to(device), pos_weight=pos_w) \
                       + l2 * lin.weight.pow(2).sum()
                opt.zero_grad(); loss.backward(); opt.step()
            if ep % 2 == 1 or ep == epochs - 1:
                p = predict(lin, Xdv); auc = safe_auc(ydv, p)
                f1 = calibrate_threshold(np.stack([1 - p, p], 1), ydv)[1]; sc = dev_score(f1, auc)
                if sc > best[0]:
                    pt = predict(lin, Xte) if Xte is not None else None
                    best = (sc, (p, pt), {'l2': l2, 'epoch': ep + 1, 'auc': auc, 'f1': f1})
        del lin, opt; free_mem()
    (p, pt), info = best[1], best[2]
    if verbose:
        print(f"  [{name}] linear probe on {D}-d features: Dev AUC {info['auc']:.4f}, F1 {info['f1']:.4f} "
              f"(l2={info['l2']}, epoch {info['epoch']}, {time.time()-t0:.0f}s)")
    to2 = lambda q: None if q is None else np.stack([1 - q, q], 1)
    return to2(p), to2(pt), info

# ---------------------------- Stage A: head on cached features ----------------
def train_head(head: nn.Module, Xtr, Mtr, ytr, Xdv, Mdv, ydv, name, soft=None, soft_mask=None):
    """soft: (N, 2) teacher probabilities at temperature T_KD, used where soft_mask is True (Run 4)."""
    head.to(device)
    mu, sd = feat_stats(Xtr, Mtr); set_head_stats(head, mu, sd)
    opt = AdamW(head.parameters(), lr=LP_LR, weight_decay=LP_WD)
    n = len(ytr); steps_per = math.ceil(n / LP_BS); total = LP_EPOCHS * steps_per; warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm)))))
    crit = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTH)
    yt = torch.as_tensor(ytr)
    best, best_state, bad, t0 = -1.0, None, 0, time.time()
    for ep in range(1, LP_EPOCHS + 1):
        head.train(); perm = torch.randperm(n); tot = 0.0
        for s in range(0, n, LP_BS):
            b = perm[s:s + LP_BS]
            logits, _ = head(Xtr[b].to(device).float(), Mtr[b].to(device))
            loss = crit(logits, yt[b].to(device))
            if soft is not None:
                sm = soft_mask[b]
                if sm.any():
                    kl = F.kl_div(F.log_softmax(logits[sm.to(device)] / T_KD, -1), soft[b][sm].to(device),
                                  reduction='batchmean') * T_KD ** 2
                    loss = (1 - KD_ALPHA) * loss + KD_ALPHA * kl * (sm.float().mean().item())
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(head.parameters(), 1.0); opt.step(); sched.step()
            tot += loss.item() * len(b)
        p = softmax_np(head_predict(head, Xdv, Mdv)[0])
        tau, f1, f1d = calibrate_threshold(p, ydv); auc = safe_auc(ydv, p[:, 1]); sc = dev_score(f1, auc)
        flag = ''
        if sc > best:
            best, bad, flag = sc, 0, ' *'
            best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
        else:
            bad += 1
        print(f"  [{name} Stage A] ep {ep:2d} | train loss {tot/n:.4f} | Dev F1@tau* {f1:.4f} (F1@0.5 {f1d:.4f}) | AUC {auc:.4f}{flag}")
        if bad >= LP_PATIENCE:
            print(f"  [{name} Stage A] early stop"); break
    head.load_state_dict(best_state)
    p = softmax_np(head_predict(head, Xdv, Mdv)[0])
    tau, f1, _ = calibrate_threshold(p, ydv)
    print(f"✓ [{name} Stage A] best Dev Macro-F1 {f1:.4f} @tau*={tau:.3f}, AUC {safe_auc(ydv, p[:, 1]):.4f} ({(time.time()-t0)/60:.1f} min)")
    return p

# ---------------------------- Stage B: PEFT fine-tuning ------------------------
def peft_groups(model: nn.Module, lr_head=FT_LR_HEAD, lr_peft=FT_LR_PEFT, e_mult: float = 1.0):
    head, peft, lam = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if n.startswith('head.'): head.append(p)
        elif 'lora_E' in n: lam.append(p)
        else: peft.append(p)
    groups = [{'params': head, 'lr': lr_head, 'weight_decay': 0.01, 'name': 'head'}]
    if peft: groups.append({'params': peft, 'lr': lr_peft, 'weight_decay': 0.0, 'name': 'peft'})
    if lam:  groups.append({'params': lam, 'lr': lr_peft * e_mult, 'weight_decay': 0.0, 'name': 'lambda'})
    return groups

def make_batch(idxs) -> dict:
    ids_l, m_l, counts = [], [], []
    for i in idxs:
        ids, m = train_store.get(int(i)); ids_l.append(ids); m_l.append(m); counts.append(ids.size(0))
    return {'idx': [int(i) for i in idxs], 'input_ids': torch.cat(ids_l).to(device), 'mask': torch.cat(m_l).to(device),
            'counts': counts, 'labels': torch.as_tensor(train_store.labels[np.asarray(idxs)], dtype=torch.long).to(device)}

def make_ce_step(criterion):
    def step(model, batch):
        logits = model(batch['input_ids'], batch['mask'], batch['counts'])
        return criterion(logits.float(), batch['labels'])
    return step

def finetune(model: DocModel, name: str, run_dir: str, param_groups, step_fn,
             doc_pool=None, after_step=None, minutes=0):
    os.makedirs(run_dir, exist_ok=True)
    if minutes <= 0:
        print(f"[{name} Stage B] skipped (FT_MINUTES=0)"); return
    doc_pool = np.arange(len(train_store)) if doc_pool is None else np.asarray(doc_pool)
    opt = AdamW(param_groups)
    for g in opt.param_groups: g['base_lr'] = g['lr']
    params = [p for g in param_groups for p in g['params']]
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    n_tr, n_all = sum(p.numel() for p in params), sum(p.numel() for p in model.parameters())
    dpf = FT_DOCS_PER_FWD
    print(f"[{name} Stage B] trainable {n_tr/1e6:.2f}M / {n_all/1e6:.1f}M params ({100*n_tr/n_all:.2f}%) | "
          f"budget {minutes} min | {dpf} docs/forward, {FT_ACCUM} docs/step | groups: " +
          ", ".join(f"{g.get('name','?')}={g['lr']:.1e}" for g in param_groups))
    def snapshot():
        return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in trainable or k.endswith('rank_mask')}
    def evaluate():
        p = softmax_np(predict_store(model, dev_store)[0])
        tau, f1, f1d = calibrate_threshold(p, dev_labels); auc = safe_auc(dev_labels, p[:, 1])
        return dev_score(f1, auc), f1, f1d, auc, tau, float((p[:, 1] >= tau).mean())
    sc, f1, f1d, auc, tau, pr = evaluate()
    print(f"[{name} Stage B] start (= Stage-A head on live features): Dev F1@tau* {f1:.4f} | AUC {auc:.4f}")
    best, best_state, bad = sc, snapshot(), 0
    scaler = make_scaler()
    budget, t0, WARM = minutes * 60, time.time(), 20
    def set_lr(step):
        frac = min(1.0, (time.time() - t0) / budget)
        s = min(1.0, (step + 1) / WARM) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * frac)))
        for g in opt.param_groups: g['lr'] = g['base_lr'] * s
    rng = np.random.RandomState(SEED + 1)
    order, pos = rng.permutation(doc_pool), 0
    docs, since, step, skipped, run_loss, next_eval, last_eval = 0, 0, 0, 0, [], FT_EVAL_EVERY, 0
    model.train(); set_lr(0); opt.zero_grad(set_to_none=True)
    while time.time() - t0 < budget:
        idxs = []
        for _ in range(dpf):
            if pos >= len(order): order, pos = rng.permutation(doc_pool), 0
            idxs.append(int(order[pos])); pos += 1
        try:
            batch = make_batch(idxs)
            with amp_ctx():
                loss = step_fn(model, batch)
            if not torch.isfinite(loss):
                skipped += len(idxs); continue
            scaler.scale(loss * len(idxs) / FT_ACCUM).backward()
        except torch.cuda.OutOfMemoryError:
            opt.zero_grad(set_to_none=True); since = 0; batch = loss = None; free_mem()
            if dpf == 1: raise
            dpf = max(1, dpf // 2); print(f"  ⚠ out of GPU memory -> {dpf} docs per forward pass"); continue
        docs += len(idxs); since += len(idxs); run_loss.append(loss.item())
        if since >= FT_ACCUM:
            scaler.unscale_(opt)
            if step == 0:   # diagnostic: gradients must reach every parameter group
                norms = []
                for g in opt.param_groups:
                    gs = [p.grad.float().norm() for p in g['params'] if p.grad is not None]
                    norms.append(f"{g.get('name','?')}={torch.stack(gs).norm().item():.2e}" if gs else f"{g.get('name','?')}=NO GRAD")
                print(f"  [{name} Stage B] first-step grad norms: " + ", ".join(norms))
            nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update()
            if after_step is not None: after_step(min(1.0, (time.time() - t0) / budget))
            opt.zero_grad(set_to_none=True); step += 1; since = 0; set_lr(step)
        if docs >= next_eval:
            next_eval += FT_EVAL_EVERY; last_eval = docs
            sc, f1, f1d, auc, tau, pr = evaluate(); model.train()
            flag = ''
            if sc > best: best, best_state, bad, flag = sc, snapshot(), 0, ' *'
            else: bad += 1
            k = max(1, FT_EVAL_EVERY // max(1, dpf))
            print(f"  [{name} Stage B] {docs} docs / {step} steps | loss {np.mean(run_loss[-k:]):.4f} | "
                  f"Dev F1@tau* {f1:.4f} (F1@0.5 {f1d:.4f}) | AUC {auc:.4f} | pos-rate {pr:.2f} | "
                  f"{(time.time()-t0)/60:.0f}/{minutes} min{flag}")
            if bad >= FT_PATIENCE:
                print(f"  [{name} Stage B] early stop"); break
    if docs > last_eval:
        sc, f1, f1d, auc, tau, pr = evaluate()
        if sc > best: best, best_state = sc, snapshot()
        print(f"  [{name} Stage B] end: {docs} docs | Dev F1@tau* {f1:.4f} | AUC {auc:.4f}")
    if skipped: print(f"  ⚠ {skipped} documents gave a non-finite loss and were skipped")
    model.load_state_dict(best_state, strict=False)
    torch.save(best_state, os.path.join(run_dir, "best_trainable.pt"))

# ---------------------------- packaging / persistence -------------------------
def package_submission(test_probs: np.ndarray, ids: List[str], threshold: float, output_dir: str, zip_filename: str):
    os.makedirs(output_dir, exist_ok=True)
    preds = (test_probs[:, 1] >= threshold).astype(int)
    json_path = os.path.join(output_dir, "prediction.json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump([{"id": cid, "pred": int(p)} for cid, p in zip(ids, preds)], f, indent=2)
    zip_path = os.path.join(WORK_DIR, zip_filename)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        verified = json.load(zf.open("prediction.json"))
    assert len(verified) == len(test_records), "Mismatch in test count!"
    c1 = sum(r['pred'] for r in verified); c0 = len(verified) - c1
    print(f"✓ {zip_path} | {len(verified)} predictions -> Class 0: {c0} ({c0/len(verified)*100:.1f}%), Class 1: {c1} ({c1/len(verified)*100:.1f}%)")
    return zip_path

def save_member(key: str, label: str, dev_probs: np.ndarray, test_probs: np.ndarray, extra: Optional[dict] = None):
    tau, f1, f1d = calibrate_threshold(dev_probs, dev_labels)
    RUN_PREDICTIONS_DEV[key], RUN_PREDICTIONS_TEST[key] = dev_probs, test_probs
    RUN_METRICS_SUMMARY[label] = {'f1': f1, 'threshold': tau}
    np.save(os.path.join(OUT_DIR, f"{key}_dev.npy"), dev_probs)
    np.save(os.path.join(OUT_DIR, f"{key}_test.npy"), test_probs)
    meta = {'key': key, 'label': label, 'f1': f1, 'threshold': tau, 'auc': safe_auc(dev_labels, dev_probs[:, 1]),
            'preproc': PREPROC, 'n_train': len(train_store)}
    meta.update(extra or {})
    with open(os.path.join(OUT_DIR, f"{key}_meta.json"), 'w') as f: json.dump(meta, f, indent=2, default=str)
    return tau, f1, f1d

def finalize_run(model: DocModel, key: str, label: str, run_dir: str, zip_name: str, extra=None):
    dl, tl = predict_store(model, dev_store, 'Dev')[0], predict_store(model, test_store, 'Test')[0]
    dev_probs, test_probs = softmax_np(dl), softmax_np(tl)
    tau, f1, f1d = save_member(key, label, dev_probs, test_probs, extra)
    print(f"{label} | tau*={tau:.3f} | Dev Macro-F1 {f1:.4f} (F1@0.5 {f1d:.4f}, AUC {safe_auc(dev_labels, dev_probs[:, 1]):.4f})")
    print(classification_report(dev_labels, (dev_probs[:, 1] >= tau).astype(int), digits=4, zero_division=0))
    package_submission(test_probs, test_ids, tau, run_dir, zip_name)

def run_lpft(key, label, run_no, run_dir, zip_name, encoder: FeatureEncoder, head: nn.Module, make_groups,
             step_fn=None, doc_pool=None, after_step=None, soft=None, soft_mask=None):
    """Frozen features -> Check 1 probe + saved doc views -> Stage A head -> Stage B PEFT -> Dev/Test predictions."""
    os.makedirs(run_dir, exist_ok=True)
    encoder.to(device)
    tag = f"{key}_{PREPROC}_{CHUNK_LEN}_{HEAD_CHUNKS}_{TAIL_CHUNKS}_{'-'.join(map(str, POOL_LAYERS))}_{len(train_store)}"
    Xtr, Mtr = extract_features(encoder, train_store, f'{key} train', cache_key=f"{tag}_train")
    Xdv, Mdv = extract_features(encoder, dev_store, f'{key} dev', cache_key=f"{tag}_dev")
    Xte, Mte = extract_features(encoder, test_store, f'{key} test', cache_key=f"{tag}_test")
    # frozen doc-level views: Check 1 now, and the fusion member of Run 5 later
    views = {sp: doc_views(X, M) for sp, (X, M) in (('train', (Xtr, Mtr)), ('dev', (Xdv, Mdv)), ('test', (Xte, Mte)))}
    for sp, v in views.items(): np.save(os.path.join(OUT_DIR, f"{key}_frozen_{sp}.npy"), v)
    pdv, pte, info = fit_linear(views['train'], train_store.labels, views['dev'], dev_labels, views['test'], name=f"{key} CHECK 1")
    save_member(f"{key}_probe", f"{label} – frozen probe", pdv, pte, info)
    del views, Xte, Mte; free_mem()
    pA = train_head(head, Xtr, Mtr, train_store.labels, Xdv, Mdv, dev_labels, key, soft=soft, soft_mask=soft_mask)
    stage_a_f1 = calibrate_threshold(pA, dev_labels)[1]
    del Xtr, Mtr, Xdv, Mdv; free_mem()
    model = DocModel(encoder, head).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTH)
    finetune(model, key, run_dir, make_groups(model), step_fn or make_ce_step(criterion),
             doc_pool=doc_pool, after_step=after_step, minutes=FT_MINUTES.get(run_no, 0))
    finalize_run(model, key, label, run_dir, zip_name, extra={'stage_a_f1': stage_a_f1, 'probe_f1': info['f1']})
    return model

print("✓ Engine ready.")

## RUN 1: Architecture 4 + LORMA  [FLAGSHIP]
Backbone `ALiBi + SwiGLU + RMSNorm` (~108M, Phase-1 PPL 23.66) · LORMA r=16 attention / r=8 FFN · BiLSTM document head

In [ ]:
# RUN 1: Architecture 4 (ALiBi + SwiGLU + RMSNorm) + LORMA + hierarchical BiLSTM head  [FLAGSHIP]
#   Stage A: frozen backbone -> chunk vectors cached once -> standardised BiLSTM head, many epochs
#   Stage B: LORMA adapters (B = 0, so the model starts exactly at the Stage-A solution) + head, jointly
#   Afterwards the fine-tuned model is the teacher for Run 4 (outputs cached to OUT_DIR/teacher_kd.pt).
if 1 in RUNS_TO_EXECUTE:
    free_mem(); AMP_ON = True
    RUN1_DIR = os.path.join(WORK_DIR, "run1_arch4_lorma")
    backbone4 = build_backbone(4, "Arch4 backbone")
    backbone_sanity(backbone4, "Arch4")
    inject_lorma(backbone4, r_attn=16, r_ffn=8, alpha=32.0, dropout=0.05)
    backbone4.to(device)
    enc4 = FeatureEncoder(backbone4)
    head1 = BiLSTMDocHead(enc4.out_dim)
    model1 = run_lpft('Run1_Arch4_LORMA', 'Run 1 (Arch 4 LORMA)', 1, RUN1_DIR, "submission_run1_arch4_lorma_BEST.zip",
                      enc4, head1, lambda m: peft_groups(m))

    print(f"Caching teacher outputs on {min(KD_DOCS, len(train_store))} training documents for Run 4 (KD)...")
    kd_idx = np.sort(np.random.RandomState(SEED).choice(len(train_store), min(KD_DOCS, len(train_store)), replace=False))
    t_logits, _ = predict_store(model1, train_store.subset(kd_idx), 'teacher')
    torch.save({'idx': kd_idx, 'logits': t_logits, 'n_train': len(train_store), 'preproc': PREPROC},
               os.path.join(OUT_DIR, "teacher_kd.pt"))
    print(f"✓ Teacher cache saved: logits {tuple(t_logits.shape)}")
    del t_logits
    del model1, enc4, head1, backbone4
    free_mem()